# <font color='blue'> Appendix A5: Proximal Policy Optimization (PPO) for Language Models </font>

In Chapter 35, we introduced **Proximal Policy Optimization (PPO)** as a powerful Reinforcement Learning algorithm.

PPO improves a policy gradually,

preventing unstable updates that could destroy previously learned behaviour.

The same idea can be applied to Large Language Models.

Instead of controlling robots or game characters,

the policy now generates text.

The objective remains the same:

improve the policy,

while ensuring that each update remains sufficiently small.

This chapter explains how PPO is adapted for Reinforcement Learning from Human Feedback (RLHF).

---



# <font color='orange'> 1. Motivation </font>

Suppose a language model has already learned

- grammar,
- reasoning,
- mathematics,
- programming.

We now wish to make it

- more helpful,
- more polite,
- more aligned with human preferences.

A naive optimization procedure might dramatically change the model,

causing it to

- forget language,
- repeat phrases,
- generate unnatural text.

PPO avoids this problem

by making only

small,

carefully controlled updates.

---



# <font color='orange'> 2. Reinforcement Learning Formulation </font>

The language model becomes

the policy.

```
Prompt

↓

Language Model

↓

Generated Response

↓

Reward Model

↓

Reward

↓

PPO Update
```

The policy gradually learns

to produce responses

that receive higher rewards.

---



# <font color='orange'> 3. What is the State? </font>

In robotics,

the state might be

```
Robot Position

Velocity

Camera Image
```

For language models,

the state is

the text generated so far.

Suppose

```
Explain gravity.
```

After generating

```
Gravity is
```

the current state becomes

```
Prompt

+

Generated Tokens
```

Every new token updates the state.

---



# <font color='orange'> 4. What is the Action? </font>

The action is simply

the next token.

Example

Current text

```
Gravity is
```

Possible actions

```
a

the

an

caused

...
```

The policy chooses

one token

from the vocabulary.

---



# <font color='orange'> 5. Episode </font>

An episode begins

with a prompt

and ends

when the response is complete.

```
Prompt

↓

Generate Tokens

↓

End-of-Sequence Token

↓

Reward
```

The complete response

receives one overall reward

from the Reward Model.

---



# <font color='orange'> 6. Reward </font>

The Reward Model evaluates

the entire response.

$$
\boxed{
r(x,y),
}
$$

where

- \(x\) is the prompt,
- \(y\) is the generated response.

Higher rewards indicate

responses that humans are expected to prefer.

---



# <font color='orange'> 7. Why PPO is Necessary </font>

Suppose

gradient ascent

directly maximized

the Reward Model.

The policy might change dramatically,

leading to

```
High Reward

↓

Poor Language Quality
```

PPO instead limits

how much

the policy changes

during each optimization step.

This preserves

fluency,

reasoning,

and factual knowledge.

---



# <font color='orange'> 8. KL-Divergence Penalty </font>

During RLHF,

the language model is compared

with the original

Supervised Fine-Tuned model.

```
Current Policy

↓

Reference Policy

↓

KL Divergence
```

Large deviations

receive a penalty.

Thus,

the optimization objective becomes

```
High Reward

+

Stay Close

to

Reference Model
```

---



# <font color='orange'> 9. PPO Objective </font>

Recall the PPO objective from Chapter 35.

Instead of allowing

arbitrarily large policy updates,

PPO clips

the probability ratio

between the new

and old policies.

This produces

stable learning,

even when optimizing

large neural networks.

---



# <font color='orange'> 10. Advantage Function </font>

PPO updates depend

upon the

**advantage**

$$
\boxed{
A(s,a).
}
$$

For language models,

the advantage measures

whether

the generated response

was better

or worse

than expected.

Positive advantage

↓

increase

the probability

of generating similar responses.

Negative advantage

↓

decrease

their probability.

---



# <font color='orange'> 11. PPO Training Pipeline </font>

The complete RLHF optimization cycle becomes

```
Prompt

↓

Language Model

↓

Response

↓

Reward Model

↓

Reward

↓

Compute Advantage

↓

PPO Update

↓

Updated Language Model
```

This process repeats

over millions of prompts.

---



# <font color='orange'> 12. Why PPO Worked Well </font>

PPO became popular because it

- is stable,
- prevents catastrophic policy updates,
- works with large neural networks,
- naturally incorporates the KL penalty.

These properties made PPO well suited

for early RLHF systems.

---



# <font color='orange'> 13. Why PPO is Expensive </font>

Unfortunately,

RLHF with PPO requires

multiple components.

```
Pretrained Model

↓

Supervised Fine-Tuning

↓

Reward Model

↓

Value Network

↓

Policy Network

↓

PPO Optimization
```

Training all these models

requires substantial computational resources.

This computational cost motivated

newer preference optimization methods,

particularly

Direct Preference Optimization (DPO).

---



# <font color='orange'> 14. Applications </font>

PPO-based RLHF has been applied to

- conversational assistants,
- coding assistants,
- summarization,
- dialogue systems,
- question answering.

It played a central role in the development of early generations of aligned Large Language Models.

---

# <font color='red'> 15. Mathematical Foundations </font>

### Policy

The language model defines

$$
\boxed{
\pi_\theta(y|x),
}
$$

where

$$
y
$$

is the generated response

and

$$
x
$$

is the prompt.

---

### Probability Ratio

The PPO update compares

the new policy

with the previous policy.

$$
\boxed{
r_t(\theta)
=
\frac{
\pi_\theta(a_t|s_t)
}{
\pi_{\theta_{\mathrm{old}}}(a_t|s_t)
}.
}
$$

For language models,

the "action"

is the generated token.

---

### PPO Clipped Objective

The optimization objective is

$$
\boxed{
L_{\mathrm{PPO}}
=
\mathbb E
\left[
\min
\left(
r_tA_t,
\,
\operatorname{clip}
(r_t,1-\epsilon,1+\epsilon)
A_t
\right)
\right].
}
$$

The clipping operation

prevents excessively large policy updates.

---

### RLHF Objective

During RLHF,

the optimization also includes

the Reward Model

and

the KL penalty.

Conceptually,

$$
\boxed{
J(\theta)
=
\mathbb E
\left[
r(x,y)
\right]
-
\beta
D_{\mathrm{KL}}
(
\pi_\theta
\|
\pi_{\mathrm{ref}}
).
}
$$

PPO approximately optimizes

this objective

while maintaining stable updates.

---



# <font color='orange'> 16. Classical PPO vs PPO for RLHF </font>

| Classical PPO | PPO for RLHF |
|:---|:---|
| Robot actions | Generated tokens |
| Environment reward | Reward Model output |
| Physical environment | Prompt–response interaction |
| State = environment observation | State = prompt + generated text |
| Policy improves behaviour | Policy improves responses |

The underlying algorithm remains the same.

Only the interpretation of the components changes.

---



# <font color='orange'> 17. Common Misconceptions </font>

### Misconception 1

> PPO teaches the language model grammar.

**False.**

Grammar and language understanding are learned primarily during pretraining. PPO mainly adjusts the model's behaviour to better align with human preferences.

---

### Misconception 2

> PPO replaces Supervised Fine-Tuning.

**False.**

PPO starts from an already well-trained Supervised Fine-Tuned model. Without this initialization, reinforcement learning would be far less stable and sample-efficient.

---

### Misconception 3

> PPO directly optimizes human feedback.

**Not exactly.**

Humans do not evaluate every generated response during PPO training. Instead, PPO optimizes the predictions of the Reward Model, which serves as a learned approximation of human preferences.

---

# <font color='purple'> 18. Conceptual Summary </font>

| Concept | Description |
|:---|:---|
| PPO | Stable policy optimization algorithm |
| Policy | Language model generating responses |
| State | Prompt together with previously generated tokens |
| Action | Next generated token |
| Reward | Predicted by the Reward Model |
| KL Penalty | Prevents excessive changes to the language model |
| Advantage | Measures whether a response was better than expected |
| RLHF | Uses PPO to align the language model with human preferences |

> **Key Insight:** PPO adapts naturally from classical reinforcement learning to language models by treating text generation as a sequential decision-making process. The language model acts as the policy, each generated token is an action, and the Reward Model provides a scalar reward estimating human preference. PPO then improves the policy through stable, incremental updates while a KL-divergence penalty ensures that the model remains close to the supervised fine-tuned policy. This combination made PPO the foundation of early Reinforcement Learning from Human Feedback systems.